# PLMD general: dataset-specific residual-hybrid benchmark

A separate dataset-specific replication of the existing PLMD study. Previous hybrid artifacts remain in their original directories; this notebook does not reproduce the exact historical weights. The PLMD test has already informed prior research, so this repeat is exploratory.

The architecture is fixed across benchmarks. Five protein-grouped outer folds generate out-of-fold predictions; an inner protein-grouped split selects each expert's checkpoint. Each development candidate completes **15 epochs without early stopping**. Short-Range Expert initialization candidates are 42, 123, and 2026, selected using inner validation only; Long-Context Expert initialization is 42. Refit durations are the median selected epochs. Fusion weights are fitted to this dataset's training OOF predictions. Threshold 0.5 is fixed.

All stages write to a new dataset-specific Drive directory and resume at completed epoch boundaries. No training or test inference occurs merely by opening this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository access

A private clone uses the read-only `GITHUB_TOKEN` Colab secret. An optional `HF_TOKEN` supports model retrieval.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt h5py
%pip uninstall -y torchao

In [ ]:
import json, torch

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU; source audit is available, training needs GPU')

## Source and baseline preflight

Authors' source location: [published Drive folder](https://drive.google.com/drive/folders/1Bcx55mlv4FKqK2XVTww8XBaNqocnP71q). Missing source files belong in `SOURCE_DIR`; the published checkpoint belongs at `MODEL_FILE`. The audit also searches the upstream Git submodule for data files. Existing PLMD, external-test, and fixed-30 artifacts are not imported as new dataset-trained models.

File availability does not establish context coverage. Training can begin only after sequence reconstruction passes its recorded coverage check.

In [ ]:
CONFIG = Path('experiment/configs/benchmarks/plmd_residual_v1.json')
BENCHMARK = 'plmd'
STUDY_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/five-benchmark-study-v1')
RUN_DIR = STUDY_ROOT / BENCHMARK / 'run'
SOURCE_DIR = STUDY_ROOT / BENCHMARK / 'source'
MODEL_FILE = SOURCE_DIR / 'DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5'
# The verified PLMD checkpoint can be reused at its existing location.
MODEL_FILE = Path('/content/drive/MyDrive/MMUbiPred-replication/artifacts/DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5')
# A checkpoint already supplied in the upstream submodule can be used directly.
if not MODEL_FILE.exists() and (REPO_ROOT / 'replication/MMUbiPred' / MODEL_FILE.name).exists():
    MODEL_FILE = REPO_ROOT / 'replication/MMUbiPred' / MODEL_FILE.name
RUN_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

def stage(name, *extra):
    run_live([sys.executable, '-u', 'experiment/scripts/run_dataset_benchmark.py', name,
              '--config', CONFIG, '--run-dir', RUN_DIR, '--source-dir', SOURCE_DIR,
              '--model-file', MODEL_FILE, *extra])

stage('audit')
audit = json.loads((RUN_DIR / 'source_audit.json').read_text())
display(audit)

## Validate implementation and prepare training context

Only training accessions are retrieved here. Published windows must match UniProt at their recorded sites. Coordinate conventions for new datasets are determined from training-window reconstruction and retained for the test split. No 257-residue windows are synthesized from short peptides.

In [ ]:
assert audit['source_preflight_passed'], 'Resolve source_audit.json blockers before preparation.'
run_live([sys.executable, '-m', 'unittest', 'discover', '-s', 'experiment/tests', '-p', 'test_dataset_benchmark.py'])
PRIOR_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
extra = ['--sequence-cache', PRIOR_CACHE] if PRIOR_CACHE.exists() else []
stage('prepare-train', *extra)
display(json.loads((RUN_DIR / 'train_preparation.json').read_text()))

## Run or resume all development folds

This cell automatically scans folds 0–4 and both experts. Completed predictions and initialization candidates are reused. Interrupted training continues after the last fully completed epoch. No manual fold variable is needed. The test partition is not loaded.

In [ ]:
stage('develop')
display(json.loads((RUN_DIR / 'development_summary.json').read_text()))

## Refit and freeze the dataset-specific hybrid

The full refit uses all eligible training sites. Duration is the integer median of the five selected epochs for each expert; Short-Range initialization uses the most frequently selected seed, with the smaller seed breaking ties. Training-derived fusion weights and threshold 0.5 are frozen with checkpoint and source hashes. A collapsed development expert blocks refitting.

In [ ]:
stage('refit')
display(json.loads((RUN_DIR / 'model_lock.json').read_text()))

## Frozen published-test evaluation

Run this section only after `model_lock.json` exists. It prepares the published test partition using the frozen coordinate rule and applies the same coverage policy. Exact training/test protein, site, and window overlaps are reported without silently changing the published split. Both models are scored on identical retained test sites.

This is a repeated historical-test analysis and remains exploratory. Comparisons use the authors' dataset-specific checkpoint. They are not comparisons to a PLMD checkpoint applied to a different training benchmark. All five outcomes should be reported; per-dataset 95% intervals are not adjusted for five-benchmark multiplicity.

In [ ]:
assert (RUN_DIR / 'model_lock.json').exists(), 'Complete and freeze refitting first.'
stage('prepare-test', '--allow-test')
display(json.loads((RUN_DIR / 'test_preparation.json').read_text()))

### Legacy Keras for the released comparator

This installs the matching legacy Keras package for the existing TensorFlow version in Colab. Baseline inference executes on CPU in a fresh subprocess.

In [ ]:
import importlib.metadata as metadata
try:
    tf_version = metadata.version('tensorflow')
    legacy_version = '.'.join(tf_version.split('.')[:2]) + '.*'
    requirement = 'tf-keras==' + legacy_version
except metadata.PackageNotFoundError:
    requirement = 'tf-keras==2.19.0'
run_live([sys.executable, '-m', 'pip', 'install', '-q', requirement])

In [ ]:
stage('evaluate', '--allow-test')
result = json.loads((RUN_DIR / 'test_results.json').read_text())
display(result['metrics'])
display(result['paired_hybrid_vs_paper'])

## Resume on another Colab account

Copy the complete `MMUbiPred-research/five-benchmark-study-v1/plmd` directory, including source files, sequence caches, fold checkpoints, optimizer/RNG states, refit artifacts, and manifests. Copy the dataset-specific H5 checkpoint if it is outside that directory. The optional `MMUbiPred-research/huggingface-cache` avoids another backbone download. Mount Drive and rerun setup, paths, and the interrupted stage. The same directory layout is required. Resumption occurs at epoch boundaries; an interrupted epoch is repeated.